# Response Time Analysis (RTA)

In [94]:
import pandas as pd
import seaborn as sns

## 1. Importing datasets

In [ ]:
url2 = 'https://raw.githubusercontent.com/JeroenGuillierme/Project-MDA/main/Data/'

aed_data = pd.read_csv(
    f'{url2}aed_df_with_distances.csv')

## 2. Outlier Detection for Response Time T3-T0

IsolationForest (for outliers/anomalies of the response time T3-T0, because some are longer than 1 day)
=> probably incorrectly filled in, in the dataset

In [ ]:
print(aed_data['T3-T0'].isna().sum())  # 16233 NaN values

In [ ]:
# Split the DataFrame into two: one with the NaN and one without in the 'T3-T0' column
# DataFrame with NaN values
aed_df_with_nan = aed_data[aed_data['T3-T0'].isna()]  
# DataFrame without NaN values
aed_df_without_nan = aed_data[~aed_data['T3-T0'].isna()]  

print('Without NaN: ', len(aed_df_without_nan))
print('With NaN: ', len(aed_df_with_nan))

In [ ]:
Time = aed_df_without_nan['T3-T0']

# IsolationForest algorithm
IsoFo = IsolationForest(n_estimators=100, contamination='auto',
                        random_state=45)  # Random state added for reproducibility
y_labels = IsoFo.fit_predict(np.array(Time).reshape(-1, 1))

# Only including the inliers
aed_df_filtered = aed_df_without_nan[y_labels == 1]  # DataFrame with inliers
discarded_rows = aed_df_without_nan[y_labels == -1]  # DataFrame with outliers

min_timedelta = discarded_rows['T3-T0'].min()  # Time deltas larger than 44.27 minutes are discarded
max_timedelta = discarded_rows['T3-T0'].max()
min_timedelta2 = aed_df_filtered['T3-T0'].min()
max_timedelta2 = aed_df_filtered['T3-T0'].max()
print(f"min response time of outliers: {min_timedelta}")  # Minimum outlier value = 44.27 minutes
print(f"max response time of outliers: {max_timedelta}")  # Maximum outlier value = 80267.83 minutes
print(f"min response time of inliers: {min_timedelta2}")  # Minimum outlier value = 0.62 minutes
print(f"max response time of inliers: {max_timedelta2}")  # Maximum outlier value = 44.17 minutes
print(f"Number of discarded rows: {len(discarded_rows)}")  # 1047
print(f"Number of filtered rows: {len(aed_df_filtered)}")  # 9296

print("\nFiltered DataFrame (Inliers):")
# print(aed_df_filtered)"""

**Total dataset without outliers**

In [ ]:
# Add rows with NaN values again:
aed_ready = pd.concat([aed_df_filtered, aed_df_with_nan], axis=0)

In [ ]:
# Plot response times
sns.histplot(data=aed_ready['T3-T0'], bins=50, log_scale=True, kde=True).set(title='Logscale of Response Times', xlabel='Log(T3-T0)') 
# Right-skewed distribution, so log scale was used.

In [ ]:
# Last check if no weird values are included in the dataset

# Get the minimum and maximum values of the 'latitude' column
min_latitude = aed_ready['Latitude'].min()
max_latitude = aed_ready['Latitude'].max()
# Get the minimum and maximum values of the 'longitude' column
min_longitude = aed_ready['Longitude'].min()
max_longitude = aed_ready['Longitude'].max()
# Get the minimum and maximum values of the response time column
min_timedelta = aed_ready['T3-T0'].min()
max_timedelta = aed_ready['T3-T0'].max()

print('Length of dataset: ', len(aed_ready))
print('Number of missing values per column: \n', print(aed_ready[['Latitude', 'Longitude', 'Intervention',
                                                                  'Eventlevel', 'T3-T0', 'EventType', 'AED', 
                                                                  'Ambulance', 'Mug']].isna().sum()))  # 1210 missing values for latitude from intervention dataset
print(f"Minimum latitude of dataset: {min_latitude}")
print(f"Maximum latitude of dataset: {max_latitude}")
print(f"Minimum longitude of dataset: {min_longitude}")
print(f"Maximum longitude of dataset: {max_longitude}")
print(f"min_timedelta of dataset: {min_timedelta}")  # Minimum outlier value = 44.27 minutes
print(f"max_timedelta of dataset: {max_timedelta}")  # Maximum outlier value = 80267.83 minutes